In [ ]:
# +----------------------------------------------------------------------------+
# | Rescue Retriever                                                           |
# | Purpose: Builds the original shelter rescue dashboard and MongoDB-backed   |
# | views.                                                                     |
# | Author: Alysha Pursley                                                     |
# | Copyright (c) 2026 Alysha Pursley.                                         |
# | Proprietary original contributions; see repository LICENSE.                                               |
# +----------------------------------------------------------------------------+

# Rescue Retriever / Grazioso Salvare dashboard
# Credentials are supplied through AAC_MONGO_USER and AAC_MONGO_PASSWORD.

import base64
from pathlib import Path

import dash_leaflet as dl
import pandas as pd
import plotly.express as px
from dash import dash_table, dcc, html
from dash.dependencies import Input, Output
from jupyter_dash import JupyterDash

from animal_shelter import AnimalShelter


WATER_BREEDS = [
    "Labrador Retriever Mix",
    "Chesapeake Bay Retriever",
    "Newfoundland",
]
MOUNTAIN_BREEDS = [
    "German Shepherd",
    "Alaskan Malamute",
    "Old English Sheepdog",
    "Siberian Husky",
    "Rottweiler",
]
DISASTER_BREEDS = [
    "Doberman Pinscher",
    "German Shepherd",
    "Golden Retriever",
    "Bloodhound",
    "Rottweiler",
]

RESCUE_QUERIES = {
    "water": {
        "animal_type": "Dog",
        "breed": {"$in": WATER_BREEDS},
        "sex_upon_outcome": "Intact Female",
        "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
    },
    "mountain": {
        "animal_type": "Dog",
        "breed": {"$in": MOUNTAIN_BREEDS},
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
    },
    "disaster": {
        "animal_type": "Dog",
        "breed": {"$in": DISASTER_BREEDS},
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300},
    },
    "reset": {},
}


def records_to_dataframe(records):
    """Convert MongoDB records to a Dash-safe DataFrame."""
    frame = pd.DataFrame.from_records(records)
    if "_id" in frame.columns:
        frame = frame.drop(columns=["_id"])
    return frame


def records_for_filter(filter_type):
    """Return records for one supported rescue filter, defaulting to reset."""
    query = RESCUE_QUERIES.get(filter_type, RESCUE_QUERIES["reset"])
    return records_to_dataframe(db.read(query))


def encode_logo():
    """Return the local Grazioso Salvare logo as a data URI."""
    logo_path = Path("Grazioso Salvare Logo.png")
    encoded = base64.b64encode(logo_path.read_bytes()).decode("ascii")
    return f"data:image/png;base64,{encoded}"


db = AnimalShelter()
df = records_for_filter("reset")

app = JupyterDash(__name__)
app.layout = html.Div([
    html.Div([
        html.A(
            html.Img(
                src=encode_logo(),
                alt="Grazioso Salvare",
                style={"height": "96px", "width": "auto"},
            ),
            href="https://www.snhu.edu/",
            target="_blank",
        ),
        html.Div([
            html.H1("Grazioso Salvare Rescue Dashboard"),
            html.P("Created by Alysha Pursley"),
        ]),
    ], style={"display": "flex", "alignItems": "center", "gap": "24px"}),
    html.Hr(),
    html.Label("Rescue filter", htmlFor="filter-type"),
    dcc.RadioItems(
        id="filter-type",
        options=[
            {"label": "Water Rescue", "value": "water"},
            {"label": "Mountain / Wilderness Rescue", "value": "mountain"},
            {"label": "Disaster / Individual Tracking", "value": "disaster"},
            {"label": "Reset", "value": "reset"},
        ],
        value="reset",
        inline=True,
    ),
    html.Hr(),
    dash_table.DataTable(
        id="datatable-id",
        columns=[
            {"name": column, "id": column, "deletable": False, "selectable": True}
            for column in df.columns
        ],
        data=df.to_dict("records"),
        row_selectable="single",
        selected_rows=[0] if not df.empty else [],
        page_action="native",
        page_size=10,
        sort_action="native",
        filter_action="native",
        style_table={"overflowX": "auto"},
        style_cell={"textAlign": "left", "minWidth": "100px", "maxWidth": "240px"},
    ),
    html.Br(),
    html.Hr(),
    html.Div(
        className="row",
        style={"display": "flex", "gap": "24px", "flexWrap": "wrap"},
        children=[
            html.Div(
                id="graph-id", className="col s12 m6", style={"flex": "1 1 480px"}
            ),
            html.Div(id="map-id", className="col s12 m6", style={"flex": "1 1 480px"}),
        ],
    ),
])


@app.callback(Output("datatable-id", "data"), Input("filter-type", "value"))
def update_dashboard(filter_type):
    """Refresh the table from MongoDB for the selected rescue category."""
    filtered = records_for_filter(filter_type)
    return filtered.to_dict("records")


@app.callback(
    Output("graph-id", "children"), Input("datatable-id", "derived_virtual_data")
)
def update_graphs(view_data):
    """Render a breed-distribution chart for the currently visible records."""
    records = view_data if view_data is not None else df.to_dict("records")
    chart_data = pd.DataFrame.from_records(records)
    if chart_data.empty or "breed" not in chart_data.columns:
        return html.P("No records are available for the current filter.")

    return dcc.Graph(
        figure=px.pie(
            chart_data,
            names="breed",
            title="Breed Distribution for Current Results",
        )
    )


@app.callback(
    Output("datatable-id", "style_data_conditional"),
    Input("datatable-id", "selected_columns"),
)
def update_styles(selected_columns):
    """Highlight user-selected table columns."""
    return [
        {"if": {"column_id": column}, "background_color": "#D2F3FF"}
        for column in (selected_columns or [])
    ]


@app.callback(
    Output("map-id", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_virtual_selected_rows"),
    ],
)
def update_map(view_data, selected_rows):
    """Render the selected animal on the Austin-area map."""
    records = view_data if view_data is not None else df.to_dict("records")
    if not records:
        return html.P("No location data are available for the current filter.")

    selected_index = selected_rows[0] if selected_rows else 0
    if selected_index < 0 or selected_index >= len(records):
        selected_index = 0

    selected = records[selected_index]
    latitude = selected.get("location_lat")
    longitude = selected.get("location_long")
    if pd.isna(latitude) or pd.isna(longitude):
        return html.P("The selected animal does not have usable location coordinates.")

    animal_name = selected.get("name") or "Unnamed"
    breed = selected.get("breed") or "Unknown breed"

    return dl.Map(
        style={"width": "100%", "height": "500px"},
        center=[latitude, longitude],
        zoom=10,
        children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(
                position=[latitude, longitude],
                children=[
                    dl.Tooltip(breed),
                    dl.Popup([
                        html.H3("Animal"),
                        html.P(f"{animal_name} - {breed}"),
                    ]),
                ],
            ),
        ],
    )


app.run_server(debug=True)
